# Preprocessing — Regex: ambil bagian penting dari teks
**Kapan dipakai:** kolom kode/nomor/email/alamat dengan format campur aduk, dan hanya sebagian
yang penting (tahun dari kode peserta, operator dari nomor HP, domain dari email, kode pos dari
alamat).

**Langkah** kerja regex:
1. Uji pola di beberapa **contoh** dulu.
2. Tulis pola dengan **grup bernama** `(?P<nama>...)`. `str.extract` langsung mengubah tiap grup
   menjadi kolom.
3. Lihat baris yang **tidak cocok**, perbaiki pola, ulangi sampai sisanya kosong atau wajar.

| Potongan | Arti |
|---|---|
| `(?P<tahun>\d{4})` | grup bernama: bagian penting, jadi kolom `tahun` |
| `(?:jl\|jln\|jalan)` | grup tanpa nama: harus cocok, tapi tidak diambil |
| `[-/. ]?` | satu karakter pemisah, boleh tidak ada |
| `\d{2,5}` / `\d+` | 2 sampai 5 digit / 1 digit atau lebih |
| `(?i)` di awal pola | abaikan huruf besar/kecil |
| `(?x)` di awal pola | boleh spasi dan komentar `#` di dalam pola (pola panjang jadi terbaca) |
| `^` `$` `\b` | awal teks, akhir teks, batas kata |
| `.*?` | apa saja, sesedikit mungkin |
| `\d+(?=\s*kg)` | lookahead: angka yang **diikuti** "kg", tanpa ikut mengambil "kg" |

Pembersihan baris per baris seperti ini tidak memakai target, jadi aman dilakukan sebelum split.

**Data demo:** `kotor_SINTETIS` (dibuat oleh `_alat/buat_data_kotor.py`).

In [1]:
import pandas as pd

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/preprocessing"   # GANTI saat lomba
DIRTY_CSV = f"{DATA_FOLDER}/kotor_SINTETIS.csv"
KEY_CSV = f"{DATA_FOLDER}/kotor_SINTETIS_kunci.csv"   # hanya ada di data latihan

# kode peserta, dua urutan: tahun di depan (2021-TI-00451) dan prodi di depan (TI-2021-451)
CODE_PATTERN_A = r"(?P<tahun>\d{4}|\d{2})[-/.]?(?P<prodi>[A-Za-z]{2})[-/.]?(?P<nomor>\d{1,5})"
CODE_PATTERN_B = r"(?P<prodi>[A-Za-z]{2})-(?P<tahun>\d{4})-(?P<nomor>\d{1,5})"
EMAIL_PATTERN = r"^(?P<user>[\w.]+)@(?P<domain>[\w.]+\.[a-z]+)$"
ADDRESS_PATTERN = r"""(?ix)
    \brt\.?\s*(?P<rt>\d{1,3})        # RT 03 / rt.3 / RT003
    \W*rw\.?\s*(?P<rw>\d{1,3})       # RW 05 / rw.5 / RW005, dipisah / , atau spasi
    .*?(?P<kota>[a-z]+)              # kata terakhir sebelum kode pos = kota
    [\s,-]*(?P<kodepos>\d{5})?       # kode pos 5 digit, boleh tidak ada
    \s*$                             # sampai akhir teks
"""

## Baca data

In [3]:
# dtype=str: semua kolom dibaca sebagai teks, supaya "0812..." tidak berubah jadi 812...
table = pd.read_csv(DIRTY_CSV, dtype=str)
print(table.shape)
table[["kode_peserta", "no_hp", "email", "alamat"]].head(10)

(3000, 10)


,kode_peserta,no_hp,email,alamat
0,AK-2024-68417,(+62) 885-7820-3289,budi@ui.ac.id.,"jln mawar no94 rt.16 rw.2 semarang, 83481"
1,2022MN98895,-,budi at itb dot ac dot id,"Jalan Ahmad Yani 164, RT019 RW007, Kab. Semara..."
2,ST-2022-66368,081529815987,Dewi99@GMAIL.COM,"Jalan Melati 108, RT015 RW007, Kab. Bandung 75410"
3,No. 2023-MN-37944,085935464656,nur@gmail.com,Melati No 11 Rt 11 Rw 7 Medan
4,No. 2023-HK-48269,0896-9952-1517,nur@yahoo.co.id.,JL SUDIRMAN NO 176 RT 11 RW 1 SURABAYA - 97254
5,MN-2021-32216,-,andi_p@student.ugm.ac.id,"Jalan Sudirman 16, RT003 RW010, Kab. Malang 57136"
6,2024-TI-41260,6289305778618,andi_p@student.ugm.ac.id,JL AHMAD YANI NO 126 RT 16 RW 14 BANDUNG - 53680
7,2024-HK-57333,0893-1209-2429,Budi@ITB.AC.ID,"Jalan Ahmad Yani 153, RT019 RW008, Kab. Padang..."
8,No. 2021-HK-74576,(+62) 827-8663-7169,nur@yahoo.co.id,"Jl. Kenanga No. 41 RT 07/RW 09, Kota Makassar ..."
9,MN-2022-30666,-,rizky.f@gmail.com.,Kenanga No 140 Rt 13 Rw 6 Bandung


## Kode peserta → tahun, prodi, nomor
Ada dua urutan: tahun di depan (`2021-TI-00451`, `2021TI00451`, `21/TI/451`, `2021.ti.00451`)
dan prodi di depan (`TI-2021-451`). Jadi dibuat dua pola (`CODE_PATTERN_A`, `CODE_PATTERN_B`).
Pola B mengisi baris yang gagal di pola A.

In [4]:
examples = pd.Series(["2021-TI-00451", "21/TI/451", "No. 2021-TI-00451", "TI-2021-451"])
example_a = examples.str.extract(CODE_PATTERN_A)
example_b = examples.str.extract(CODE_PATTERN_B)
print(example_a.fillna(example_b))

  tahun prodi  nomor
0  2021    TI  00451
1    21    TI    451
2  2021    TI  00451
3  2021    TI    451


In [5]:
code_a = table["kode_peserta"].str.extract(CODE_PATTERN_A)
code_b = table["kode_peserta"].str.extract(CODE_PATTERN_B)
code = code_a.fillna(code_b)
year = pd.to_numeric(code["tahun"])
code["tahun"] = year.where(year > 100, year + 2000)      # 21 -> 2021
code["prodi"] = code["prodi"].str.upper()
code["nomor"] = pd.to_numeric(code["nomor"])

not_matched = code.isna().any(axis=1)
print("tidak cocok:", not_matched.sum())
print(table.loc[not_matched, "kode_peserta"].head())
code.head()

tidak cocok: 0
Series([], Name: kode_peserta, dtype: object)


,tahun,prodi,nomor
0,2024,AK,68417
1,2022,MN,98895
2,2022,ST,66368
3,2023,MN,37944
4,2023,HK,48269


## Nomor HP → format seragam `08...` + operator
Lebih mudah **membuang** yang tidak perlu daripada mencocokkan semua bentuk:
buang semua selain digit, ganti `62` di depan dengan `0`, lalu cek bentuk akhirnya.

In [6]:
phone = table["no_hp"].str.replace(r"\D", "", regex=True)    # buang semua selain digit
phone = phone.str.replace(r"^62", "0", regex=True)             # 62812... -> 0812...
phone = phone.where(phone.str.fullmatch(r"08\d{8,11}"))        # bukan nomor HP -> NaN
phone_prefix = phone.str[:4]                                   # 0812, 0857, ... (operator)

print("kosong / bukan HP:", phone.isna().sum())
pd.DataFrame({"mentah": table["no_hp"], "bersih": phone, "prefix": phone_prefix}).head(8)

kosong / bukan HP: 383


,mentah,bersih,prefix
0,(+62) 885-7820-3289,088578203289,0885
1,-,NaN,NaN
2,081529815987,081529815987,0815
3,085935464656,085935464656,0859
4,0896-9952-1517,089699521517,0896
5,-,NaN,NaN
6,6289305778618,089305778618,0893
7,0893-1209-2429,089312092429,0893


## Email → user, domain, email kampus
Perbaiki dulu bentuk yang ditulis manusia (`budi at gmail dot com`, spasi, titik di akhir),
baru ambil bagiannya.

In [7]:
email = table["email"].str.strip()
email = email.str.lower()
email = email.str.replace(r"\s+at\s+", "@", regex=True)
email = email.str.replace(r"\s+dot\s+", ".", regex=True)
email = email.str.rstrip(".")
email_parts = email.str.extract(EMAIL_PATTERN)
email_parts["kampus"] = email_parts["domain"].str.endswith(".ac.id")

print("tidak cocok:", email_parts["domain"].isna().sum())
email_parts.head()

tidak cocok: 0


,user,domain,kampus
0,budi,ui.ac.id,True
1,budi,itb.ac.id,True
2,dewi99,gmail.com,False
3,nur,gmail.com,False
4,nur,yahoo.co.id,False


## Alamat → RT, RW, kota, kode pos
Pola panjang (`ADDRESS_PATTERN` di Konfigurasi) ditulis dengan `(?x)`, satu bagian per baris,
masing-masing dengan komentar.

In [8]:
address = table["alamat"].str.extract(ADDRESS_PATTERN)
address["rt"] = pd.to_numeric(address["rt"])
address["rw"] = pd.to_numeric(address["rw"])
address["kota"] = address["kota"].str.title()

print("tidak cocok:", address["kota"].isna().sum())
print("tanpa kode pos:", address["kodepos"].isna().sum())
pd.concat([table["alamat"], address], axis=1).head(8)

tidak cocok: 0
tanpa kode pos: 591


,alamat,rt,rw,kota,kodepos
0,"jln mawar no94 rt.16 rw.2 semarang, 83481",16,2,Semarang,83481
1,"Jalan Ahmad Yani 164, RT019 RW007, Kab. Semara...",19,7,Semarang,66629
2,"Jalan Melati 108, RT015 RW007, Kab. Bandung 75410",15,7,Bandung,75410
3,Melati No 11 Rt 11 Rw 7 Medan,11,7,Medan,NaN
4,JL SUDIRMAN NO 176 RT 11 RW 1 SURABAYA - 97254,11,1,Surabaya,97254
5,"Jalan Sudirman 16, RT003 RW010, Kab. Malang 57136",3,10,Malang,57136
6,JL AHMAD YANI NO 126 RT 16 RW 14 BANDUNG - 53680,16,14,Bandung,53680
7,"Jalan Ahmad Yani 153, RT019 RW008, Kab. Padang...",19,8,Padang,76848


## Bonus: semua kemunculan dan lookahead
`str.findall` mengambil **semua** kecocokan (hasilnya list). Lookahead `(?=...)` mensyaratkan
teks sesudahnya tanpa ikut mengambilnya.

In [9]:
first_addresses = table["alamat"].head(3)
print(list(first_addresses.str.findall(r"\d+")))                    # semua angka di alamat
print(first_addresses.str.extract(r"(?i)no\.?\s*(\d+)(?=\D)"))      # nomor rumah sesudah "No"

[['94', '16', '2', '83481'], ['164', '019', '007', '66629'], ['108', '015', '007', '75410']]
     0
0   94
1  NaN
2  NaN


## Simpan hasil

In [10]:
result = pd.DataFrame()
result["id"] = table["id"]
result["tahun_masuk"] = code["tahun"]
result["kode_prodi"] = code["prodi"]
result["nomor_urut"] = code["nomor"]
result["no_hp"] = phone
result["prefix_hp"] = phone_prefix
result["email"] = email
result["domain"] = email_parts["domain"]
result["email_kampus"] = email_parts["kampus"]
result["rt"] = address["rt"]
result["rw"] = address["rw"]
result["kota"] = address["kota"]
result["kodepos"] = address["kodepos"]
result.to_csv("hasil_regex.csv", index=False)
result.head()

,id,tahun_masuk,kode_prodi,nomor_urut,no_hp,prefix_hp,email,domain,email_kampus,rt,rw,kota,kodepos
0,1,2024,AK,68417,088578203289,0885,budi@ui.ac.id,ui.ac.id,True,16,2,Semarang,83481
1,2,2022,MN,98895,NaN,NaN,budi@itb.ac.id,itb.ac.id,True,19,7,Semarang,66629
2,3,2022,ST,66368,081529815987,0815,dewi99@gmail.com,gmail.com,False,15,7,Bandung,75410
3,4,2023,MN,37944,085935464656,0859,nur@gmail.com,gmail.com,False,11,7,Medan,NaN
4,5,2023,HK,48269,089699521517,0896,nur@yahoo.co.id,yahoo.co.id,False,11,1,Surabaya,97254


## Cek dengan kunci jawaban latihan (HAPUS saat lomba)
Angka dibandingkan sebagai teks: "2021.0" dan "2021" dianggap sama.

In [11]:
key = pd.read_csv(KEY_CSV, dtype=str)
for column in ["tahun_masuk", "kode_prodi", "nomor_urut", "no_hp", "email", "rt", "rw", "kota",
               "kodepos"]:
    guess = result[column].astype(str).str.removesuffix(".0")
    answer = key[column].astype(str).str.removesuffix(".0")
    correct = guess == answer
    print(f"{column:12s} benar {correct.mean():.3f}")

tahun_masuk  benar 1.000
kode_prodi   benar 1.000
nomor_urut   benar 1.000
no_hp        benar 1.000
email        benar 1.000
rt           benar 1.000
rw           benar 1.000
kota         benar 1.000
kodepos      benar 1.000
